# CI Payment Details Monthly Burn EDA

This notebook analyzes `ci_payment_details.csv`, focusing on the `_months` model fields as requested. `_pg` fields are retained in the inventory but are not used for primary interpretation.

## Brief Analysis and Assumptions

The dataset has 9,042 posting rows for 363 contract items.

Every field is populated in this extract; missingness is not the main issue.

The monthly delta formulas are internally consistent: 9,042 absolute-delta rows and 9,042 percent-delta rows match the stated formulas.

The monthly linear model is a poor row-level predictor for many postings: median `BURN_DELTA_MONTHS_PERCENT` is -66.42%, and only 1,174 rows are within +/-25% of linear.

The distribution is asymmetric and bursty: 822 rows exceed +100%, while 157 rows are below -100%.

Negative actual burn exists in 157 rows, so reversals/corrections materially affect the lower tail.

Only 165 of 363 items have row count equal to `NUM_PAYGROUPS`; only 192 have distinct posting-date count equal to `NUM_PAYGROUPS`. This is the main grain caveat.

The item-level monthly rate reconciles to item total burn for 0 of 363 items using `(DAYS_BETWEEN + 30) / 30`, confirming the broad monthly-rate construction.

Clarifying questions carried as assumptions for this pass:

- Should repeated `ITEMID` + `WPPOSTINGDATE` rows be collapsed to one pay group, or retained as separate posting/detail rows? This notebook retains them and separately reports duplicate date groups.
- Should negative `THIS_BURN` values be treated as valid correction/reversal postings? This notebook treats them as valid and profiles their impact.
- Is `THIS_BURN` an amount for the posting row rather than a pre-normalized monthly rate? This notebook interprets it as the actual row burn compared to the monthly-linear model.

In [ ]:
import csv
from collections import Counter, defaultdict
from decimal import Decimal
from datetime import datetime
from pathlib import Path

CSV_PATH = Path('ci_payment_details.csv')
with CSV_PATH.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
    fields = reader.fieldnames

print(f'Rows: {len(rows):,}')
print(f'Fields: {len(fields):,}')
print(fields)

## Field Inventory

| Field | Role | Missing | Missing % | Unique | Examples | Interpretation |
| --- | --- | --- | --- | --- | --- | --- |
| ITEMID | identifier/count | 0 | 0.00% | 363 | 17074 | Contract item identifier. User described each ITEMID as a particular contract item with multiple payment postings. |
| WPPOSTINGDATE | date | 0 | 0.00% | 2,601 | 2017-08-16 06:00:00.000; 2017-03-08 07:00:00.000; 2017-03-23 06:00:00.000 | Posting date for this actual burn observation. |
| FIRSTPOSTINGDATE | date | 0 | 0.00% | 271 | 2017-02-28 07:00:00.000 | First posting date observed for the ITEMID. |
| LASTPOSTINGDATE | date | 0 | 0.00% | 265 | 2017-11-10 07:00:00.000 | Last posting date observed for the ITEMID. |
| NUM_PAYGROUPS | identifier/count | 0 | 0.00% | 30 | 8 | Total number of postings/pay groups for the ITEMID according to the query. |
| DAYS_BETWEEN | identifier/count | 0 | 0.00% | 250 | 285 | Days between first and last posting for the ITEMID. |
| CI_BURN_RATE_PG | pay-group model | 0 | 0.00% | 359 | 151229.520000000000 | Linear item burn rate using fixed pay-group units. Ignored for primary interpretation in this notebook. |
| CI_BURN_RATE_MONTHS | monthly model | 0 | 0.00% | 363 | 127351.174736842105 | Linear item burn rate using elapsed months: roughly item total burn / ((DAYS_BETWEEN + 30) / 30). |
| THIS_BURN | actual burn | 0 | 0.00% | 7,792 | 66674.46240000; 17376.32520000; 63427.09230000 | Actual burn amount/rate for this row. |
| BURN_DELTA_MONTHS_PERCENT | monthly model | 0 | 0.00% | 7,937 | -47.645192486200; -86.355583106400; -50.195125854900 | Monthly-model normalized deviation: (THIS_BURN - CI_BURN_RATE_MONTHS) / CI_BURN_RATE_MONTHS * 100. |
| BURN_DELTA_PG_PERCENT | pay-group model | 0 | 0.00% | 7,565 | -55.911741041000; -88.509964721200; -58.059053351500 | Pay-group-model normalized deviation. Ignored for primary interpretation in this notebook. |
| BURN_DELTA_PG | pay-group model | 0 | 0.00% | 7,995 | -84555.057600000000; -133853.194800000000; -87802.427700000000 | Absolute delta from pay-group linear model. Ignored for primary interpretation in this notebook. |
| BURN_DELTA_MONTHS | monthly model | 0 | 0.00% | 8,017 | -60676.712336842105; -109974.849536842105; -63924.082436842105 | Absolute delta from monthly linear model: THIS_BURN - CI_BURN_RATE_MONTHS. |

## Numeric Ranges

| Field | Parsed | Missing | Parse failures | Min | P05 | P25 | Median | P75 | P95 | Max | Sum | Mean |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| ITEMID | 9,042 | 0 | 0 | 17,074.0000 | 19,681.0000 | 40,650.0000 | 56,847.0000 | 89,754.0000 | 101,292.0000 | 112,628.0000 | 565,937,804.00 | 62,589.8921 |
| NUM_PAYGROUPS | 9,042 | 0 | 0 | 7.0000 | 7.0000 | 8.0000 | 11.0000 | 15.0000 | 32.0000 | 46.0000 | 123,119.00 | 13.6163 |
| DAYS_BETWEEN | 9,042 | 0 | 0 | 137.0000 | 246.0000 | 394.0000 | 562.0000 | 791.0000 | 1,201.0000 | 1,402.0000 | 5,686,196.00 | 628.8649 |
| CI_BURN_RATE_MONTHS | 9,042 | 0 | 0 | 11,463.6981 | 25,729.1668 | 39,069.8081 | 81,868.7602 | 251,661.0777 | 2,203,947.3684 | 10,226,006.4134 | 3,674,934,595.55 | 406,429.3957 |
| THIS_BURN | 9,042 | 0 | 0 | -6,313,204.5000 | 698.7746 | 7,773.0754 | 35,286.2500 | 120,291.9900 | 1,201,221.2837 | 30,600,018.7090 | 3,077,710,874.20 | 340,379.4375 |
| BURN_DELTA_MONTHS_PERCENT | 9,042 | 0 | 0 | -738.6050 | -98.9631 | -88.2496 | -66.4210 | -8.2203 | 176.0000 | 1,834.6319 | -238,193.33 | -26.3430 |
| BURN_DELTA_MONTHS | 9,042 | 0 | 0 | -9,982,342.4134 | -780,752.9899 | -112,636.2058 | -33,607.7404 | -4,111.8427 | 253,595.6293 | 23,159,344.9328 | -597,223,721.35 | -66,049.9581 |
| CI_BURN_RATE_PG | 9,042 | 0 | 0 | 31,038.3723 | 46,135.2381 | 68,963.4000 | 123,313.4372 | 393,451.5200 | 2,233,333.3333 | 11,094,405.4519 | 4,456,955,849.49 | 492,917.0371 |
| BURN_DELTA_PG_PERCENT | 9,042 | 0 | 0 | -543.9500 | -99.3323 | -92.4223 | -79.0636 | -37.4436 | 75.0000 | 1,177.5278 | -477,800.00 | -52.8423 |
| BURN_DELTA_PG | 9,042 | 0 | 0 | -10,850,741.4519 | -802,375.3525 | -207,525.2174 | -67,119.0542 | -32,190.8411 | 170,114.9028 | 23,040,725.5497 | -1,379,244,975.29 | -152,537.5996 |

## Date Ranges

| Field | Parsed | Missing | Parse failures | Min | Median | Max |
| --- | --- | --- | --- | --- | --- | --- |
| WPPOSTINGDATE | 9,042 | 0 | 0 | 2017-02-28 07:00:00 | 2021-06-18 06:00:00 | 2026-04-30 06:00:00 |
| FIRSTPOSTINGDATE | 9,042 | 0 | 0 | 2017-02-28 07:00:00 | 2020-06-27 06:00:00 | 2025-07-23 06:00:00 |
| LASTPOSTINGDATE | 9,042 | 0 | 0 | 2017-10-26 06:00:00 | 2022-07-29 06:00:00 | 2026-04-30 06:00:00 |

## Internal Consistency Checks

| Check | Pass | Fail |
| --- | --- | --- |
| BURN_DELTA_MONTHS = THIS_BURN - CI_BURN_RATE_MONTHS | 9,042 | 0 |
| BURN_DELTA_MONTHS_PERCENT formula | 9,042 | 0 |
| WPPOSTINGDATE within FIRST/LAST span | 9,042 | 0 |
| FIRSTPOSTINGDATE <= LASTPOSTINGDATE | 9,042 | 0 |
| DAYS_BETWEEN agrees with FIRST/LAST dates | 0 | 9,042 |

Exact duplicate rows: 101. Duplicate `ITEMID` + `WPPOSTINGDATE` groups: 624.

## Row Grain and Paygroup Count

`NUM_PAYGROUPS` behaves like an item-level count from the query rather than a guaranteed count of rows in this export. This matters because repeated dates or row-level detail splits can make a posting group appear more than once.

| NUM_PAYGROUPS | Item count | Share of items |
| --- | --- | --- |
| 7 | 87 | 23.97% |
| 8 | 62 | 17.08% |
| 9 | 32 | 8.82% |
| 10 | 31 | 8.54% |
| 11 | 31 | 8.54% |
| 12 | 17 | 4.68% |
| 13 | 22 | 6.06% |
| 14 | 13 | 3.58% |
| 15 | 14 | 3.86% |
| 16 | 5 | 1.38% |
| 17 | 4 | 1.10% |
| 18 | 4 | 1.10% |
| 19 | 5 | 1.38% |
| 20 | 4 | 1.10% |
| 21 | 2 | 0.55% |
| 22 | 2 | 0.55% |
| 23 | 1 | 0.28% |
| 24 | 4 | 1.10% |
| 25 | 1 | 0.28% |
| 26 | 5 | 1.38% |
| 27 | 3 | 0.83% |
| 29 | 1 | 0.28% |
| 31 | 1 | 0.28% |
| 32 | 2 | 0.55% |
| 34 | 1 | 0.28% |
| 35 | 3 | 0.83% |
| 37 | 1 | 0.28% |
| 38 | 3 | 0.83% |
| 42 | 1 | 0.28% |
| 46 | 1 | 0.28% |

Items where row count equals `NUM_PAYGROUPS`: 165 of 363.

Items where distinct posting dates equal `NUM_PAYGROUPS`: 192 of 363.

## Monthly Delta Distribution

| Delta percent bucket | Rows | Share |
| --- | --- | --- |
| < -100% | 157 | 1.74% |
| -100% to -75% | 3,713 | 41.06% |
| -75% to -50% | 1,575 | 17.42% |
| -50% to -25% | 861 | 9.52% |
| -25% to +25% | 1,174 | 12.98% |
| +25% to +100% | 740 | 8.18% |
| > +100% | 822 | 9.09% |

The large mass below zero means most posting rows are below the even monthly allocation. The positive tail indicates concentrated/bursty postings that exceed the linear monthly model by multiples.

## Delta by Number of Paygroups

| NUM_PAYGROUPS bucket | Rows | P25 delta % | Median delta % | P75 delta % | P95 delta % | Total this burn | Negative burn share |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 07-08 | 2,462 | -83.89 | -65.52 | -14.41 | 205.00 | 263,123,840.28 | 1.50% |
| 09-12 | 2,820 | -90.64 | -71.22 | -21.19 | 176.11 | 218,536,675.19 | 1.38% |
| 13-18 | 2,210 | -91.53 | -71.26 | -20.15 | 164.17 | 162,225,423.34 | 1.58% |
| 19-30 | 1,012 | -83.46 | -58.38 | 1.28 | 144.87 | 736,755,913.00 | 3.85% |
| 31+ | 538 | -83.53 | -10.25 | 35.11 | 135.49 | 1,697,069,022.40 | 1.30% |

## Delta by Item Duration

| DAYS_BETWEEN bucket | Rows | P25 delta % | Median delta % | P75 delta % | P95 delta % | Total this burn | Negative burn share |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1095+ | 801 | -87.20 | -53.78 | 35.66 | 219.17 | 1,593,108,095.31 | 2.12% |
| 180-364 | 1,744 | -86.79 | -68.13 | -26.18 | 119.33 | 184,923,420.30 | 2.47% |
| 365-729 | 4,116 | -89.22 | -68.57 | -10.03 | 195.48 | 396,894,873.73 | 0.85% |
| 730-1094 | 2,295 | -86.88 | -62.63 | 0.87 | 179.32 | 888,061,084.43 | 2.70% |
| <180 | 86 | -89.47 | -88.05 | -82.91 | -51.15 | 14,723,400.42 | 0.00% |

## Top Positive Monthly Delta Rows

These are the posting rows most above the monthly-linear model.

| ITEMID | WPPOSTINGDATE | NUM_PAYGROUPS | DAYS_BETWEEN | CI_BURN_RATE_MONTHS | THIS_BURN | BURN_DELTA_MONTHS_PERCENT | BURN_DELTA_MONTHS |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 19672 | 2017-05-06 06:00:00.000 | 15 | 942 | 107615.573248407643 | 2081965.25000000 | 1834.631937697600 | 1974349.676751592357 |
| 19670 | 2017-05-06 06:00:00.000 | 12 | 942 | 100318.471337579618 | 1922666.51000000 | 1816.562806793700 | 1822348.038662420382 |
| 61751 | 2020-05-15 06:00:00.000 | 10 | 1234 | 64916.348500132484 | 1229991.00000000 | 1794.732264550400 | 1165074.651499867516 |
| 44983 | 2019-11-16 07:00:00.000 | 11 | 905 | 39701.768378985985 | 692580.00000000 | 1644.456300759100 | 652878.231621014015 |
| 57300 | 2020-07-27 06:00:00.000 | 9 | 1293 | 196851.902552204176 | 3381031.00000000 | 1617.550583034600 | 3184179.097447795824 |
| 68867 | 2021-08-21 06:00:00.000 | 8 | 758 | 26145.418388582871 | 424836.73469800 | 1524.899354770000 | 398691.316309417129 |
| 75397 | 2022-03-19 06:00:00.000 | 13 | 561 | 39326.763636363636 | 630099.69926400 | 1502.216000000000 | 590772.935627636364 |
| 80920 | 2022-04-16 06:00:00.000 | 7 | 520 | 123351.811795227150 | 1974747.34050800 | 1500.906635880000 | 1851395.528712772850 |
| 71522 | 2021-05-21 06:00:00.000 | 14 | 1234 | 146645.057914465623 | 2179270.00000000 | 1386.084857541600 | 2032624.942085534377 |
| 36218 | 2019-01-18 07:00:00.000 | 11 | 1349 | 11463.698073619733 | 166792.03300000 | 1354.958355749300 | 155328.334926380267 |
| 73938 | 2021-08-21 06:00:00.000 | 13 | 723 | 47238.506224066390 | 681816.50720000 | 1343.349000000000 | 634578.000975933610 |
| 57300 | 2020-09-25 06:00:00.000 | 9 | 1293 | 196851.902552204176 | 2738048.20000000 | 1290.917824263300 | 2541196.297447795824 |
| 63638 | 2021-05-15 06:00:00.000 | 38 | 1192 | 167406.551823880468 | 2236218.42000000 | 1235.801015931900 | 2068811.868176119532 |
| 87872 | 2023-08-28 06:00:00.000 | 9 | 612 | 69283.854166666667 | 769635.90000000 | 1010.844523961700 | 700352.045833333333 |
| 36921 | 2019-08-16 06:00:00.000 | 12 | 673 | 58218.213673376132 | 630584.03110000 | 983.138748704600 | 572365.817426623868 |
| 19669 | 2017-05-06 06:00:00.000 | 14 | 942 | 70700.636942675159 | 762775.17000000 | 978.880195405400 | 692074.533057324841 |
| 64963 | 2020-11-21 07:00:00.000 | 13 | 632 | 36966.051155600456 | 389375.74500000 | 953.333350000000 | 352409.693844399544 |
| 19671 | 2017-05-06 06:00:00.000 | 15 | 942 | 78025.477707006369 | 796647.17000000 | 921.009026040800 | 718621.692292993631 |
| 51375 | 2019-10-26 06:00:00.000 | 15 | 1201 | 18109.908560448864 | 181250.00000000 | 900.833325000000 | 163140.091439551136 |
| 87855 | 2023-08-25 06:00:00.000 | 7 | 802 | 60893.880123365089 | 599103.75000000 | 883.848867548400 | 538209.869876634911 |

## Top Negative Monthly Delta Rows

These rows are most below the monthly-linear model. Values below -100% are possible because actual burn can be negative.

| ITEMID | WPPOSTINGDATE | NUM_PAYGROUPS | DAYS_BETWEEN | CI_BURN_RATE_MONTHS | THIS_BURN | BURN_DELTA_MONTHS_PERCENT | BURN_DELTA_MONTHS |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 75397 | 2022-01-15 07:00:00.000 | 13 | 561 | 39326.763636363636 | -251142.67892000 | -738.605000000000 | -290469.442556363636 |
| 57300 | 2023-09-12 06:00:00.000 | 9 | 1293 | 196851.902552204176 | -851893.50000000 | -532.758580920500 | -1048745.402552204176 |
| 19679 | 2018-05-19 06:00:00.000 | 27 | 898 | 121830.157704122023 | -511373.00000000 | -519.742541286000 | -633203.157704122023 |
| 26988 | 2018-02-27 07:00:00.000 | 8 | 329 | 48126.106135984616 | -165981.00000000 | -444.887657295600 | -214107.106135984616 |
| 58603 | 2020-11-09 07:00:00.000 | 12 | 851 | 57101.153265556366 | -178174.63400000 | -412.033337000000 | -235275.787265556366 |
| 17383 | 2018-07-21 06:00:00.000 | 15 | 594 | 69641.467171717172 | -188903.64600000 | -371.251674715900 | -258545.113171717172 |
| 63537 | 2021-01-20 07:00:00.000 | 7 | 321 | 51401.869158878505 | -137500.00000000 | -367.500000000000 | -188901.869158878505 |
| 41320 | 2020-03-12 06:00:00.000 | 9 | 432 | 77356.082222222222 | -202736.01600000 | -362.081545724600 | -280092.098222222222 |
| 36218 | 2022-03-07 07:00:00.000 | 11 | 1349 | 11463.698073619733 | -29444.69063500 | -356.851588779700 | -40908.388708619733 |
| 57754 | 2021-01-23 07:00:00.000 | 32 | 1225 | 2659138.013543983784 | -6313204.50000000 | -337.415450715400 | -8972342.513543983784 |
| 17423 | 2018-12-27 07:00:00.000 | 19 | 667 | 26524.546400667862 | -61564.23980000 | -332.102893938500 | -88088.786200667862 |
| 44999 | 2020-09-29 06:00:00.000 | 18 | 980 | 24002.399755077554 | -48333.60000000 | -301.369865068500 | -72335.999755077554 |
| 92749 | 2025-03-14 06:00:00.000 | 15 | 530 | 420033.527546537216 | -727604.86000000 | -273.225424229800 | -1147638.387546537216 |
| 32544 | 2019-02-15 07:00:00.000 | 21 | 1031 | 28191.270337621044 | -44200.00000000 | -256.786123756200 | -72391.270337621044 |
| 31342 | 2018-05-21 06:00:00.000 | 7 | 195 | 122775.853846153846 | -189319.41000000 | -254.199220831500 | -312095.263846153846 |
| 89748 | 2025-09-02 06:00:00.000 | 15 | 746 | 34457.886937561837 | -50063.70000000 | -245.289524255300 | -84521.586937561837 |
| 31342 | 2018-09-06 06:00:00.000 | 7 | 195 | 122775.853846153846 | -158187.33000000 | -228.842378240100 | -280963.183846153846 |
| 45014 | 2020-04-17 06:00:00.000 | 15 | 809 | 22792.042208998242 | -27670.50000000 | -221.404215323300 | -50462.542208998242 |
| 32539 | 2019-03-15 06:00:00.000 | 37 | 1129 | 2560043.508769207341 | -2980400.00000000 | -216.419896372500 | -5540443.508769207341 |
| 47606 | 2020-01-07 07:00:00.000 | 11 | 356 | 42817.170145585108 | -47897.40000000 | -211.864936045800 | -90714.570145585108 |

## Largest Items by Absolute Total Burn

| ITEMID | Rows | Unique posting dates | NUM_PAYGROUPS | DAYS_BETWEEN | Modeled months | CI burn rate months | Sum this burn | Min delta % | Median delta % | Max delta % | Negative rows |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 68698 | 42 | 42 | 42 | 1367 | 46.57 | 10,226,006.41 | 465,965,028.98 | -97.62 | 21.06 | 185.61 | 0 |
| 54059 | 46 | 46 | 46 | 1402 | 47.73 | 7,440,673.78 | 347,727,485.33 | -98.13 | -3.96 | 311.25 | 0 |
| 93523 | 29 | 29 | 29 | 948 | 32.60 | 6,831,868.47 | 215,887,043.60 | -97.63 | 6.73 | 136.40 | 0 |
| 42512 | 36 | 35 | 35 | 1219 | 41.63 | 5,201,006.13 | 211,334,214.00 | -100.00 | -4.38 | 229.41 | 0 |
| 29914 | 38 | 38 | 38 | 1256 | 42.87 | 4,369,703.52 | 182,944,921.96 | -101.69 | 22.01 | 118.15 | 1 |
| 96813 | 29 | 26 | 26 | 791 | 27.37 | 6,299,591.15 | 166,099,222.07 | -108.20 | -3.15 | 101.48 | 3 |
| 57754 | 35 | 32 | 32 | 1225 | 41.83 | 2,659,138.01 | 108,581,468.00 | -337.42 | -18.21 | 212.30 | 2 |
| 94749 | 26 | 26 | 26 | 809 | 27.97 | 3,919,235.93 | 105,688,730.22 | -91.59 | -10.79 | 142.96 | 0 |
| 32539 | 73 | 37 | 37 | 1129 | 38.63 | 2,560,043.51 | 96,342,969.86 | -216.42 | -91.08 | 135.24 | 3 |
| 44374 | 31 | 31 | 31 | 947 | 32.57 | 3,040,983.39 | 95,993,710.00 | -99.68 | -7.83 | 137.38 | 0 |
| 71520 | 58 | 34 | 34 | 1234 | 42.13 | 2,223,418.27 | 91,456,604.00 | -100.00 | -69.90 | 191.52 | 0 |
| 29913 | 38 | 38 | 38 | 1249 | 42.63 | 943,787.04 | 39,293,000.00 | -94.68 | -63.94 | 572.50 | 0 |
| 96809 | 26 | 26 | 26 | 790 | 27.33 | 1,320,664.57 | 34,777,500.00 | 1.28 | 1.28 | 1.28 | 0 |
| 96810 | 15 | 15 | 15 | 456 | 16.20 | 2,203,947.37 | 33,500,000.00 | -99.56 | 1.47 | 87.27 | 0 |
| 19674 | 33 | 26 | 26 | 898 | 30.93 | 916,290.70 | 27,427,634.54 | -120.85 | -57.90 | 215.93 | 3 |
| 19677 | 25 | 24 | 24 | 843 | 29.10 | 966,075.64 | 27,146,725.59 | -99.24 | 9.02 | 201.98 | 0 |
| 19675 | 27 | 26 | 26 | 898 | 30.93 | 892,689.37 | 26,721,168.31 | -151.64 | -72.11 | 533.43 | 1 |
| 19676 | 29 | 27 | 27 | 898 | 30.93 | 762,356.61 | 22,819,874.37 | -99.71 | -36.40 | 213.40 | 0 |
| 63936 | 28 | 27 | 27 | 1030 | 35.33 | 627,981.19 | 21,560,687.45 | -100.00 | -0.78 | 665.72 | 0 |
| 29911 | 35 | 35 | 35 | 1059 | 36.30 | 566,572.24 | 20,000,000.00 | 0.85 | 0.85 | 0.87 | 0 |
| 71521 | 32 | 32 | 32 | 968 | 33.27 | 581,962.80 | 18,778,000.00 | -21.62 | -21.62 | 239.65 | 0 |
| 106167 | 17 | 17 | 17 | 546 | 19.20 | 808,177.26 | 14,708,826.21 | -95.92 | 23.10 | 223.98 | 0 |
| 43164 | 77 | 77 | 7 | 397 | 14.23 | 1,071,380.67 | 14,177,937.16 | -98.01 | -82.45 | -68.62 | 0 |
| 91118 | 11 | 11 | 11 | 618 | 21.60 | 635,119.71 | 13,083,466.00 | -94.85 | 49.35 | 406.35 | 0 |
| 96814 | 19 | 19 | 19 | 638 | 22.27 | 603,291.53 | 12,830,000.00 | -99.78 | -33.51 | 368.63 | 0 |

## Most Volatile Items by Mean Absolute Delta Percent

| ITEMID | Rows | Unique posting dates | NUM_PAYGROUPS | DAYS_BETWEEN | Sum this burn | Mean abs delta % | Min delta % | Median delta % | Max delta % | Negative rows |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 57300 | 9 | 9 | 9 | 1293 | 8,484,317.00 | 497.97 | -532.76 | 212.60 | 1,617.55 | 1 |
| 61751 | 11 | 11 | 10 | 1234 | 2,670,225.78 | 347.89 | -206.72 | 217.41 | 1,794.73 | 2 |
| 57455 | 8 | 8 | 7 | 1053 | 612,500.00 | 346.20 | -29.80 | 251.00 | 847.70 | 0 |
| 89262 | 7 | 7 | 7 | 882 | 1,400,000.00 | 320.00 | 47.00 | 194.00 | 635.00 | 0 |
| 69763 | 7 | 7 | 7 | 856 | 990,000.00 | 307.62 | 42.67 | 185.33 | 613.33 | 0 |
| 69765 | 7 | 7 | 7 | 856 | 505,000.00 | 307.62 | 42.67 | 185.33 | 613.33 | 0 |
| 80920 | 7 | 7 | 7 | 520 | 2,138,098.03 | 281.21 | -96.19 | -77.47 | 1,500.91 | 0 |
| 87855 | 8 | 8 | 7 | 802 | 1,627,896.38 | 274.29 | -119.74 | 88.39 | 883.85 | 1 |
| 68867 | 8 | 8 | 8 | 758 | 660,607.58 | 270.37 | -81.81 | 13.32 | 1,524.90 | 0 |
| 71522 | 14 | 14 | 14 | 1234 | 6,032,000.00 | 250.14 | -99.93 | 12.21 | 1,386.08 | 0 |
| 19670 | 12 | 12 | 12 | 942 | 3,150,000.00 | 239.47 | -95.02 | -21.75 | 1,816.56 | 0 |
| 91200 | 8 | 8 | 8 | 812 | 700,000.00 | 238.33 | 35.33 | 170.67 | 576.67 | 0 |
| 21019 | 8 | 8 | 8 | 718 | 512,753.41 | 237.20 | -100.00 | 175.23 | 546.20 | 0 |
| 44983 | 11 | 11 | 11 | 905 | 1,197,670.03 | 228.17 | -75.72 | 27.42 | 1,644.46 | 0 |
| 57298 | 7 | 7 | 7 | 643 | 7,349,420.21 | 228.13 | -76.80 | 122.55 | 617.84 | 0 |
| 75397 | 14 | 13 | 13 | 561 | 735,410.48 | 210.91 | -738.61 | -28.75 | 1,502.22 | 2 |
| 32534 | 13 | 13 | 13 | 1149 | 2,500,000.00 | 210.00 | -100.00 | 219.17 | 219.17 | 0 |
| 90940 | 7 | 7 | 7 | 526 | 8,321,776.19 | 205.38 | -88.25 | 114.61 | 426.00 | 0 |
| 89271 | 7 | 7 | 7 | 639 | 532,380.00 | 204.29 | 30.97 | 208.35 | 397.44 | 0 |
| 58603 | 13 | 12 | 12 | 851 | 1,619,769.40 | 203.64 | -412.03 | 98.57 | 609.17 | 1 |
| 19669 | 14 | 14 | 14 | 942 | 2,220,000.00 | 200.42 | -150.68 | -16.19 | 978.88 | 1 |
| 19672 | 18 | 15 | 15 | 942 | 3,379,129.00 | 200.37 | -137.90 | -86.41 | 1,834.63 | 3 |
| 19681 | 7 | 7 | 7 | 520 | 544,711.60 | 192.76 | -83.87 | 25.45 | 682.36 | 0 |
| 87870 | 8 | 8 | 8 | 593 | 1,325,281.00 | 191.07 | -78.51 | 68.04 | 520.94 | 0 |
| 36923 | 8 | 8 | 8 | 499 | 922,363.60 | 190.34 | -117.86 | -37.79 | 548.57 | 1 |

## Negative and Zero Burn Rows

| Metric | Value |
| --- | --- |
| Negative THIS_BURN rows | 157 |
| Zero THIS_BURN rows | 59 |
| Items with at least one negative row | 70 |

### Items With Most Negative Rows

| ITEMID | Negative rows | Negative row share | Item total this burn |
| --- | --- | --- | --- |
| 92684 | 17 | 16.67% | 656,267.70 |
| 44916 | 14 | 25.00% | 1,581,307.20 |
| 19679 | 7 | 18.42% | 3,646,782.68 |
| 47606 | 7 | 9.59% | 508,097.10 |
| 47641 | 6 | 8.82% | 583,597.50 |
| 39911 | 5 | 8.77% | 3,581,741.20 |
| 44978 | 5 | 3.40% | 10,422,319.14 |
| 31342 | 4 | 28.57% | 798,043.05 |
| 80683 | 4 | 3.74% | 615,680.00 |
| 19672 | 3 | 16.67% | 3,379,129.00 |
| 19674 | 3 | 9.09% | 27,427,634.54 |
| 26988 | 3 | 17.65% | 527,782.98 |
| 26998 | 3 | 10.00% | 1,551,473.60 |
| 32539 | 3 | 4.11% | 96,342,969.86 |
| 36912 | 3 | 10.71% | 797,585.83 |
| 47626 | 3 | 13.04% | 1,292,566.00 |
| 96813 | 3 | 10.34% | 166,099,222.07 |
| 19671 | 2 | 11.76% | 2,450,000.00 |
| 28398 | 2 | 5.71% | 2,489,953.33 |
| 31170 | 2 | 8.70% | 4,122,840.90 |
| 31343 | 2 | 6.67% | 2,432,447.75 |
| 45014 | 2 | 6.90% | 614,625.41 |
| 45139 | 2 | 11.11% | 576,894.50 |
| 57754 | 2 | 5.71% | 108,581,468.00 |
| 61751 | 2 | 18.18% | 2,670,225.78 |

## Posting-Date Spacing

| Metric | Count | Min | Median | Max | Mean |
| --- | --- | --- | --- | --- | --- |
| Inter-posting gap days | 8,679 | 0.00 | 5.98 | 1,025.04 | 21.77 |

### Duplicate ITEMID + WPPOSTINGDATE Groups

| ITEMID | WPPOSTINGDATE | Rows on same date |
| --- | --- | --- |
| 80683 | 2022-09-28 06:00:00.000 | 26 |
| 44916 | 2022-03-18 06:00:00.000 | 20 |
| 44980 | 2019-09-06 06:00:00.000 | 17 |
| 82692 | 2023-02-18 07:00:00.000 | 17 |
| 82692 | 2023-05-27 06:00:00.000 | 14 |
| 80683 | 2022-05-06 06:00:00.000 | 13 |
| 44980 | 2019-09-18 06:00:00.000 | 12 |
| 82692 | 2023-01-21 07:00:00.000 | 10 |
| 82692 | 2023-04-29 06:00:00.000 | 9 |
| 92684 | 2024-11-14 07:00:00.000 | 9 |
| 101203 | 2024-12-16 20:58:41.000 | 9 |
| 20931 | 2017-10-27 06:00:00.000 | 8 |
| 44980 | 2019-09-04 06:00:00.000 | 8 |
| 80683 | 2022-05-05 06:00:00.000 | 8 |
| 44980 | 2019-10-19 06:00:00.000 | 7 |
| 92684 | 2025-06-05 06:00:00.000 | 7 |
| 92752 | 2025-05-17 06:00:00.000 | 7 |
| 20941 | 2017-10-19 06:00:00.000 | 6 |
| 20941 | 2017-11-16 07:00:00.000 | 6 |
| 32539 | 2019-03-15 06:00:00.000 | 6 |
| 45014 | 2020-05-15 06:00:00.000 | 6 |
| 46472 | 2019-10-16 06:00:00.000 | 6 |
| 51364 | 2020-04-17 06:00:00.000 | 6 |
| 80683 | 2022-05-19 06:00:00.000 | 6 |
| 80683 | 2022-09-29 06:00:00.000 | 6 |
| 92754 | 2024-12-13 07:00:00.000 | 6 |
| 19679 | 2018-11-17 07:00:00.000 | 5 |
| 20941 | 2018-01-09 07:00:00.000 | 5 |
| 20968 | 2018-09-14 06:00:00.000 | 5 |
| 30704 | 2018-08-18 06:00:00.000 | 5 |

## Modeling Implications

- The monthly-linear model is useful as a baseline, but it should not be expected to predict individual posting rows closely. The empirical pattern is under-burn in many periods and sharp over-burn in fewer concentrated periods.
- `BURN_DELTA_MONTHS_PERCENT` is a better cross-item comparability field than the absolute delta, but it still explodes for low baseline rates and for negative/correction rows. Winsorized or bucketed views will likely be more stable than raw percent deltas for model training.
- Treat `ITEMID` as the primary grouping key. Any train/test split or validation should avoid splitting rows from the same item across train and test if the goal is generalization to unseen items.
- Before modeling payment timing, decide whether repeated `ITEMID` + `WPPOSTINGDATE` rows are separate observations or should be collapsed to a date/paygroup level. That choice changes the interpretation of `NUM_PAYGROUPS` and row weights.
- Negative burns should be modeled or flagged explicitly; excluding them would remove genuine correction behavior and would also hide why the percent delta can fall below -100%.